# BCC Preprocessing (two-pass chunked)

**Before running:**
1. Upload `GSE123813_bcc_scRNA_counts.txt` to Google Drive
2. Upload `GSE123813_bcc_tcell_metadata.txt` directly to `/content/`
3. Run all cells in order
4. Download `cleaned_GSE123813.txt` when done

Estimated time: 20-40 minutes

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Imports and paths
import pandas as pd
import numpy as np

INPUT_FILE  = '/content/drive/MyDrive/GSE123813_bcc_scRNA_counts.txt'
TCELL_META  = '/content/GSE123813_bcc_tcell_metadata.txt'
OUTPUT_FILE = '/content/cleaned_GSE123813.txt'

GENE_PREFIXES_TO_REMOVE = ('MT-', 'RPS', 'RPL', 'MRP', 'MTRNR')
MIN_CELL_PERCENT = 0.03
TOTAL_CELLS = 53030   # known from header inspection
CHUNKSIZE   = 2000

min_cells = int(np.ceil(MIN_CELL_PERCENT * TOTAL_CELLS))
print(f'Min cells threshold: {min_cells}  (3% of {TOTAL_CELLS})')

In [ ]:
# Cell 3: Load T cell IDs
meta      = pd.read_csv(TCELL_META, sep='\t')
tcell_ids = set(meta['cell.id'].astype(str).str.strip().tolist())
print(f'T cell IDs in metadata: {len(tcell_ids)}')

In [ ]:
# Cell 4: Pass 1 - count expressed cells per gene across all 53k cells
# (after removing MT/ribosomal genes)
print('Pass 1: counting expressed cells per gene...')
gene_counts = {}
i = 0

for chunk in pd.read_csv(INPUT_FILE, sep='\t', index_col=0, chunksize=CHUNKSIZE):
    # remove MT/ribosomal genes
    mask  = ~chunk.index.str.startswith(GENE_PREFIXES_TO_REMOVE)
    chunk = chunk[mask]

    # count cells where gene is expressed (value > 0)
    expressed = (chunk > 0).sum(axis=1)
    for gene, count in expressed.items():
        gene_counts[gene] = gene_counts.get(gene, 0) + count

    i += 1
    print(f'  Chunk {i} processed', end='\r')

print(f'\nTotal genes after MT/ribosomal filter: {len(gene_counts)}')

# identify genes passing 3% threshold
keep_genes = set(g for g, c in gene_counts.items() if c >= min_cells)
print(f'Genes passing 3% threshold: {len(keep_genes)}')

In [ ]:
# Cell 5: Pass 2 - read again, keep passing genes and T cell columns only
print('Pass 2: building filtered matrix...')
chunks     = []
tcell_cols = None
i          = 0

for chunk in pd.read_csv(INPUT_FILE, sep='\t', index_col=0, chunksize=CHUNKSIZE):
    # identify T cell columns on first chunk
    if tcell_cols is None:
        tcell_cols = [c for c in chunk.columns if c in tcell_ids]
        print(f'T cell columns found: {len(tcell_cols)}')

    # keep only genes that passed both filters
    chunk = chunk[chunk.index.isin(keep_genes)]

    # keep only T cell columns
    chunk = chunk[tcell_cols]

    if len(chunk) > 0:
        chunks.append(chunk)

    i += 1
    print(f'  Chunk {i} processed', end='\r')

print(f'\nConcatenating...')
df = pd.concat(chunks)
print(f'Final shape: {df.shape}  (genes x T cells)')

In [ ]:
# Cell 6: Save
print('Saving...')
df.to_csv(OUTPUT_FILE, sep='\t')
print(f'Saved: {OUTPUT_FILE}')
print(f'Final shape: {df.shape}  (genes x T cells)')
print('Done. Download cleaned_GSE123813.txt')